In [20]:
!pip install transformers

In [21]:
from transformers import pipeline

In [22]:
classifier = pipeline('sentiment-analysis')

classifier(
    [
        "I've been waiting for the AI course my whole life.",
        "You're one of the worst person I ever met",
    ]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9801843762397766},
 {'label': 'NEGATIVE', 'score': 0.9997346997261047}]

In [23]:
# Automatically converting tokens to integers
from transformers import AutoTokenizer

In [24]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokizer = AutoTokenizer.from_pretrained(checkpoint)

In [25]:
print(tokizer)

BertTokenizer(name_or_path='distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [26]:
raw_inputs = [
    "I've been waiting for a AI course my whole life.",
    "You're the worst person I have ever seen.",
]

inputs = tokizer(raw_inputs, padding=True, truncation=True, return_tensors= "pt")
print(inputs)

{'input_ids': tensor([[ 101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 9932, 2607, 2026, 2878,
         2166, 1012,  102],
        [ 101, 2017, 1005, 2128, 1996, 5409, 2711, 1045, 2031, 2412, 2464, 1012,
          102,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])}


In [27]:
# Going through the model
from transformers import AutoModel
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 15, 768])


In [30]:
print(outputs["last_hidden_state"])

tensor([[[-1.5740e-01,  4.3127e-01, -1.8119e-01,  ..., -3.6100e-01,
           8.8545e-02,  3.9843e-01],
         [ 9.0102e-02,  9.4411e-01, -1.7303e-01,  ..., -2.2004e-01,
          -1.1321e-01,  3.6780e-01],
         [ 6.4638e-01,  2.7118e-01, -5.2401e-02,  ...,  4.4022e-02,
          -4.9741e-01, -2.9692e-01],
         ...,
         [-8.4762e-02,  7.2786e-01, -1.7753e-01,  ..., -6.7663e-01,
          -1.3313e-01,  2.0710e-01],
         [ 4.7416e-01,  1.9870e-01, -3.0473e-01,  ...,  1.4744e-01,
          -3.7777e-01, -2.7799e-01],
         [-7.6767e-04,  9.1829e-01, -1.6334e-01,  ...,  4.1895e-06,
          -1.9660e-01, -1.8796e-01]],

        [[-7.9749e-01,  5.9055e-01,  1.0359e-01,  ...,  1.3742e-01,
          -8.6151e-01, -3.3846e-01],
         [-7.8758e-01,  5.6914e-01,  2.2784e-01,  ...,  1.3269e-01,
          -6.2811e-01, -4.0306e-01],
         [-6.4020e-01,  5.6959e-01,  3.4247e-01,  ..., -7.9786e-02,
          -7.1661e-01, -5.0002e-01],
         ...,
         [-4.6657e-01,  5

In [31]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [32]:
print(outputs.logits.shape)

torch.Size([2, 2])


In [33]:
print(outputs.logits)

tensor([[ 2.0602, -1.7511],
        [ 4.5484, -3.6217]], grad_fn=<AddmmBackward0>)


In [35]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[9.7836e-01, 2.1641e-02],
        [9.9972e-01, 2.8290e-04]], grad_fn=<SoftmaxBackward0>)


In [36]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}